# AI and Global Workforce Reductions
### Research Question
How do workforce reductions differ between AI-focused and non-AI-focused companies, and how are these differences related to industry, company size, geographic location, and time?

## Data Collection and Processing
Clean, merge, transform, and prepare the dataset with Python.

We are using four CSV files from the Kaggle dataset: company layoffs, U.S. labor indicators, global labor indicators, and news sentiment.

We will clean the datasets separately because they contain different types of observations. We will create summaries for comparison, but we will not merge the datasets unless a specific analysis requires it.


## 1. Import libraries and load data

In [3]:
import pandas as pd

layoffs = pd.read_csv("layoffs_events.csv")
us_labor = pd.read_csv("us_labor_indicators.csv")
global_labor = pd.read_csv("global_labor_indicators.csv")
news = pd.read_csv("news_sentiment.csv")

print("All four datasets loaded.")
print("Layoff events:", len(layoffs))
layoffs.head(10)

All four datasets loaded.
Layoff events: 2470


,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company
0,Panda Squad,SF Bay Area,6.0,2020-03-13,75%,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,$1.00,United States,False
1,HopSkipDrive,Los Angeles,8.0,2020-03-13,10%,Transportat…,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,$45.00,United States,False
2,Help.com,Austin,16.0,2020-03-16,100%,Support,LinkedIn,Seed,$6.00,United States,False
3,Service,Los Angeles,NaN,2020-03-16,100%,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,$5.00,United States,False
4,Inspirato,Denver,130.0,2020-03-16,22%,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,$79.00,United States,False
5,Flytedesk,Boulder,4.0,2020-03-18,20%,Marketing,https://businessden.com/2020 /03/18/boulder-ad...,Seed,$4.00,United States,False
6,Tuft & Needle,Phoenix,NaN,2020-03-19,NaN,Retail,https://www.theverge.com/20 20/3/19/21185840/t...,Acquired,$0.00,United States,False
7,Popin,New York City,NaN,2020-03-19,100%,Fitness,https://www.businessinsider.c om/fitness-app-p...,Unknown,$13.00,United States,False
8,Anyvision,Tel Aviv Non-U.S.,NaN,2020-03-19,NaN,Security,https://ipvm.com/reports/anyv ision-20-layoffs,Series A,$74.00,Israel,True
9,Lola,Boston,34.0,2020-03-19,NaN,Travel,https://layoffs.fyi/2020/03/19/l ola-lays-off-...,Series C,$81.00,United States,False


## 2. Inspect the datasets

Check each dataset's size, missing values, data types, and duplicate rows.

In [4]:
datasets = {
    "Layoffs": layoffs,
    "U.S. Labor": us_labor,
    "Global Labor": global_labor,
    "News Sentiment": news
}

for name, df in datasets.items():
    print("\n---", name, "---")
    print("Rows and columns:", df.shape)
    print("Exact duplicates:", df.duplicated().sum())
    print("Missing values:\n", df.isna().sum())
    print("Data types:\n", df.dtypes)


--- Layoffs ---
Rows and columns: (2470, 11)
Exact duplicates: 0
Missing values:
 company            0
location           0
layoff_count     878
date               0
pct_workforce    936
industry           2
source_url         1
stage              5
raised_mm        291
country            2
is_ai_company      0
dtype: int64
Data types:
 company           object
location          object
layoff_count     float64
date              object
pct_workforce     object
industry          object
source_url        object
stage             object
raised_mm         object
country           object
is_ai_company       bool
dtype: object

--- U.S. Labor ---
Rows and columns: (396, 10)
Exact duplicates: 0
Missing values:
 date                         0
unemployment_rate            0
jolts_job_openings_k         0
initial_jobless_claims_k     1
information_sector_emp_k     0
computer_math_emp_k          0
financial_emp_k              0
openings_per_unemployed      0
tech_emp_yoy_pct            12
claims_

## 3. Clean the layoff dataset
Remove exact duplicates if present, standardize text labels, convert dates and numeric columns, and create variables that will help with comparisons.

- `year` contains the year of each layoff event.
- `month` contains both the year and month.
- `ai_group` gives the original AI classification a clear label.

Missing values will be handled in another cell

In [7]:
layoffs = layoffs.drop_duplicates().copy()

# Remove extra spaces from text columns
for col in ["company", "location", "industry", "stage", "country"]:
    layoffs[col] = layoffs[col].astype("string").str.strip()

# Standardize shortened labels found in the supplied CSV
layoffs["country"] = layoffs["country"].replace({
    "United Kingdo…": "United Kingdom",
    "United Arab E…": "United Arab Emirates",
    "UAE": "United Arab Emirates"
})
layoffs["industry"] = layoffs["industry"].replace({
    "Transportat…": "Transportation",
    "Infrastructu…": "Infrastructure",
    "Manufactur…": "Manufacturing"
})

# Convert dates and numeric values
layoffs["date"] = pd.to_datetime(layoffs["date"], errors="coerce")
layoffs["layoff_count"] = pd.to_numeric(layoffs["layoff_count"], errors="coerce")
layoffs["pct_workforce"] = pd.to_numeric(
    layoffs["pct_workforce"].astype("string").str.replace("%", "", regex=False),
    errors="coerce"
)
layoffs["raised_mm"] = pd.to_numeric(
    layoffs["raised_mm"].astype("string").str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)

# Create useful columns for future analysis
layoffs["year"] = layoffs["date"].dt.year
layoffs["month"] = layoffs["date"].dt.to_period("M").astype(str)
layoffs["ai_group"] = layoffs["is_ai_company"].map({
    True: "AI-focused", False: "Non-AI-focused"
})

layoffs.head()

,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company,year,month,ai_group
0,Panda Squad,SF Bay Area,6.0,2020-03-13,75,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,1.0,United States,False,2020,2020-03,Non-AI-focused
1,HopSkipDrive,Los Angeles,8.0,2020-03-13,10,Transportation,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,45.0,United States,False,2020,2020-03,Non-AI-focused
2,Help.com,Austin,16.0,2020-03-16,100,Support,LinkedIn,Seed,6.0,United States,False,2020,2020-03,Non-AI-focused
3,Service,Los Angeles,NaN,2020-03-16,100,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,5.0,United States,False,2020,2020-03,Non-AI-focused
4,Inspirato,Denver,130.0,2020-03-16,22,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,79.0,United States,False,2020,2020-03,Non-AI-focused


## 4. Prepare the other three datasets
We will clean the U.S. labor, global labor, and news sentiment datasets. We will remove exact duplicates if present, format dates, and ensure appropriate units are used throughout.

For the U.S. labor dataset, convert unemployment claim counts into thousands to have consistent units.

Keep these datasets separate from the main layoff dataset because they contain different types of information. They can be merged later if needed for our analysis.

In [8]:
# U.S. labor indicators
us_labor = us_labor.drop_duplicates().copy()
us_labor["date"] = pd.to_datetime(us_labor["date"], errors="coerce")
us_labor["initial_jobless_claims_k"] = us_labor["initial_jobless_claims_k"] / 1000
us_labor["claims_4w_avg"] = us_labor["claims_4w_avg"] / 1000

# Global labor indicators
global_labor = global_labor.drop_duplicates().copy()
global_labor["year"] = pd.to_numeric(global_labor["year"], errors="coerce").astype("Int64")
global_labor["country_name"] = global_labor["country_name"].astype("string").str.strip()

# News sentiment
news = news.drop_duplicates().copy()
news["date"] = pd.to_datetime(news["date"], errors="coerce")
news["published_at"] = pd.to_datetime(
    news["published_at"], format="mixed", errors="coerce", utc=True
)

print("U.S. Labor:", us_labor.shape)
print("Global Labor:", global_labor.shape)
print("News Sentiment:", news.shape)
news[["date", "published_at"]].head()

U.S. Labor: (396, 10)
Global Labor: (114, 6)
News Sentiment: (613, 10)


,date,published_at
0,2022-11-09,2022-11-09 00:00:00+00:00
1,2022-11-09,2022-11-09 00:00:00+00:00
2,2022-11-14,2022-11-14 00:00:00+00:00
3,2022-12-07,2022-12-07 00:00:00+00:00
4,2022-12-21,2022-12-21 00:00:00+00:00


## 5. Handle missing values
Different columns need different treatment:

- **Layoffs:** We will keep missing employee counts, workforce percentages, and funding amounts as `NaN` so that we don't distort results. We will fill missing industry, country, and funding-stage labels with `"Unknown"` so those events remain available for group comparisons.
- **U.S. labor:** There are only a few missing values pertaining to hisotrical lebor indicators. We will preserve these missing values as they are not necessary for out analysis.
- **Global labor:** No missing values were found in the original CSV.
- **News sentiment:** Replace missing descriptions with empty strings and leave missing article URLs as `NaN`.

We also add a flag showing which layoff events have an unknown employee count fro future analysis.

In [9]:
# Fill categorical values without guessing the missing category
for col in ["industry", "country", "stage"]:
    layoffs[col] = layoffs[col].fillna("Unknown")

# Keep unknown layoff counts as NaN, but record which events are missing counts
layoffs["layoff_count_missing"] = layoffs["layoff_count"].isna()

# Missing descriptions are optional text
news["description"] = news["description"].fillna("")

# Check what is still missing after cleaning
cleaned_datasets = {
    "Layoffs": layoffs,
    "U.S. Labor": us_labor,
    "Global Labor": global_labor,
    "News Sentiment": news
}

for name, df in cleaned_datasets.items():
    print("\n---", name, "remaining missing values ---")
    print(df.isna().sum()[lambda counts: counts > 0])


--- Layoffs remaining missing values ---
layoff_count     878
pct_workforce    936
source_url         1
raised_mm        291
dtype: int64

--- U.S. Labor remaining missing values ---
initial_jobless_claims_k     1
tech_emp_yoy_pct            12
claims_4w_avg                4
dtype: int64

--- Global Labor remaining missing values ---
Series([], dtype: int64)

--- News Sentiment remaining missing values ---
url    37
dtype: int64


## 6. Check missing layoff counts by AI classification
Missing employee counts can affect comparisons of how many employees were laid off. We will calculate the percentage of events with missing counts for each AI group.

In [10]:
ai_missing_summary = (
    layoffs.groupby("ai_group")["layoff_count"]
    .agg(total_events="size", events_with_known_counts="count")
)
ai_missing_summary["events_missing_counts"] = (
    ai_missing_summary["total_events"] - ai_missing_summary["events_with_known_counts"]
)
ai_missing_summary["percent_missing"] = (
    ai_missing_summary["events_missing_counts"] / ai_missing_summary["total_events"] * 100
).round(1)

ai_missing_summary

,total_events,events_with_known_counts,events_missing_counts,percent_missing
ai_group,,,,
AI-focused,355,217,138,38.9
Non-AI-focused,2115,1375,740,35.0


## 7. Create Summaries for Analysis

We will create three summaries from the cleaned layoff dataset to help compare workforce reductions across time, industries, and geographic locations in future analysis.

- **Monthly Layoffs:** Groups layoffs by month and AI classification to examine trends over time.
- **Industry Layoffs:** Groups layoffs by industry and AI classification to compare workforce reductions across industries.
- **Country Layoffs:** Groups layoffs by country and AI classification to examine geographic differences.

Each summary includes the number of reported layoff events, the number of employees laid off where counts are available, and the number of events with known employee counts.

These summaries will help simplify the analysis and visualization process while preserving the original cleaned dataset.

In [11]:
# Monthly layoffs by AI classification
monthly_layoffs = (
    layoffs.groupby(["month", "ai_group"], as_index=False)
    .agg(
        layoff_events=("company", "size"),
        known_layoffs=("layoff_count", lambda x: x.sum(min_count=1)),
        events_with_known_counts=("layoff_count", "count")
    )
)

# Industry layoffs by AI classification
industry_layoffs = (
    layoffs.groupby(["industry", "ai_group"], as_index=False)
    .agg(
        layoff_events=("company", "size"),
        known_layoffs=("layoff_count", lambda x: x.sum(min_count=1)),
        events_with_known_counts=("layoff_count", "count")
    )
)

# Country layoffs by AI classification
country_layoffs = (
    layoffs[layoffs["country"] != "Unknown"]
    .groupby(["country", "ai_group"], as_index=False)
    .agg(
        layoff_events=("company", "size"),
        known_layoffs=("layoff_count", lambda x: x.sum(min_count=1)),
        events_with_known_counts=("layoff_count", "count")
    )
)

# Display summaries
print("Monthly Layoffs:")
display(monthly_layoffs.head())

print("Industry Layoffs:")
display(industry_layoffs.head())

print("Country Layoffs:")
display(country_layoffs.head())

Monthly Layoffs:


,month,ai_group,layoff_events,known_layoffs,events_with_known_counts
0,2020-03,AI-focused,5,260.0,4
1,2020-03,Non-AI-focused,114,9273.0,87
2,2020-04,AI-focused,16,323.0,7
3,2020-04,Non-AI-focused,108,14896.0,83
4,2020-05,AI-focused,5,87.0,2


Industry Layoffs:


,industry,ai_group,layoff_events,known_layoffs,events_with_known_counts
0,AI,AI-focused,16,765.0,11
1,Aerospace,AI-focused,2,NaN,0
2,Aerospace,Non-AI-focused,12,3205.0,11
3,Construction,Non-AI-focused,13,950.0,5
4,Consumer,AI-focused,18,24535.0,11


Country Layoffs:


,country,ai_group,layoff_events,known_layoffs,events_with_known_counts
0,Argentina,AI-focused,2,180.0,2
1,Argentina,Non-AI-focused,4,143.0,3
2,Australia,AI-focused,6,219.0,5
3,Australia,Non-AI-focused,39,3077.0,20
4,Austria,AI-focused,1,270.0,1


## 8. Validate and export the prepared datasets
Check that conversions work properly. Export the cleaned data and summaries to CSV files

In [15]:
# Validate cleaned data
assert layoffs["date"].notna().all()
assert us_labor["date"].notna().all()
assert news["published_at"].notna().all()

assert layoffs["layoff_count"].dropna().ge(0).all()
assert layoffs["pct_workforce"].dropna().between(0, 100).all()
assert layoffs["ai_group"].notna().all()

print("All validation checks passed.")

# Save cleaned datasets
layoffs.to_csv("layoffs_clean.csv", index=False)
us_labor.to_csv("us_labor_clean.csv", index=False)
global_labor.to_csv("global_labor_clean.csv", index=False)
news.to_csv("news_clean.csv", index=False)

# Save summaries
monthly_layoffs.to_csv("monthly_layoffs.csv", index=False)
industry_layoffs.to_csv("industry_layoffs.csv", index=False)
country_layoffs.to_csv("country_layoffs.csv", index=False)

print("Layoff events:", len(layoffs))
print("Monthly summary rows:", len(monthly_layoffs))
print("Industry summary rows:", len(industry_layoffs))
print("Country summary rows:", len(country_layoffs))

All validation checks passed.
Layoff events: 2470
Monthly summary rows: 128
Industry summary rows: 51
Country summary rows: 84


## 9. Dataset Selection for Analysis

Although we cleaned all four datasets, we plan to focus our analysis on the datasets most relevant to our research question.

- **Layoff Events (Primary Dataset):** We will use this dataset to compare reported layoffs between AI-focused and non-AI-focused companies across industries, countries, and time. The three summaries will support these comparisons.

- **Global Labor Indicators (Supporting Dataset):** This dataset may be used to explore relationships between layoffs and unemployment rates across countries. If needed, we can merge the data using country and year.

We do not currently plan to use the other two datasets in our main analysis:

- **U.S. Labor Indicators:** This dataset provides U.S.-specific labor statistics, while our primary research question focuses on global workforce reductions.

- **News Sentiment:** This dataset measures news coverage and sentiment rather than actual workforce reductions.